# Apparent Magnitudes

### Rieke neatly states....

Assuming we do not want to fight twenty centuries of tradition, we will express the results in the magnitude system. In general the magnitude difference, $m_1 - m_2$, between objects 1 and 2 in the same photometric band is...

\begin{equation}
m_1 - m_2 = -2.5 {\rm log_{10}} \left( F_1 / F_2 \right)
\end{equation}

where $F_1$ and $F_2$ are respectively the fluxes of objects 1 and 2 in the band. See Equation 5.2 in Rieke. Note that in astronomy log is base 10 by default. 

Photometric band and band refer to a wavelength range, often defined with a glass filter.

The mostly commonly used visible light filters are the five UBVRI filters and the five ugriz filters. 

In the Vega based system (e.g. UBVRI) magnitude zero is defined using the Vega, so 

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( F_1 / F_{Vega} \right)
\end{equation}

In the AB system (e.g. ugriz) magnitude zero is defined using a hypothetical $f_\nu = 3631~{\rm Jy}$ source , so 

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( F_1 / F_{3631} \right)
\end{equation}

For suitably boring spectra (e.g. smooth continuum) we find 

\begin{equation}
m_1 - m_2 \simeq -2.5 {\rm log} \left( f_{\nu,1} / f_{\nu,2} \right)
\end{equation}

\begin{equation}
m_1 - m_2 \simeq -2.5 {\rm log} \left( f_{\lambda,1} / f_{\lambda,2} \right)
\end{equation}

where $\nu$ and $\lambda$ are close to the central frequency and wavelength respectively of the photometric band or filter. 

###

# Absolute magnitudes 

### Absolute magnitudes are defined by an $m=0$ source at a distance of 10 parsec (not 1 parsec)

### A parsec is the distance where 1 au subtends an angle of 1 arcsecond (a 3600th of a degree)

### Vega can define $m=0$ but it is not an $M=0$ star as it is not at $10~{\rm pc}$ disance

### By definition 

\begin{equation}
m - M = 5 {\rm log} \left( \frac{d}{10~{\rm pc}} \right) 
\end{equation}

where $m$ is the apparent magnitude, $M$ is the absolute magnitude and $d$ is the distance. See Equation 5.5 in Rieke.

###

# <font color='red'> Magnitudes are tricky </font>

### They are based on Ancient Greek star classes where 1 is bright and 5 is faint 

### Magnitudes get numerically bigger as objects get fainter 

#### Say a brighter or fainter magnitude (not bigger or smaller) to avoid confusion

### Magnitudes are wavelength dependent 

### Except for visual magnitudes, we should define the filter we are using (e.g. $m_R$)

#### At the teaching telescopes we use UBVRI filters that are similar to the standard UBVRI filters 

### We can define a colour by subtracting magnitudes (e.g. $m_V - m_R$) as this is equivalent to a flux ratio

### Colours, like magnitudes, are backwards

#### $m_B - m_V<0$ is very blue while $m_B - m_V>2$ is very red 

#### $m_B - m_V=0$ is not white - it is blue as Vega is a blue star

###

###

# Filters

### With modern imaging devices we are typically observing celestial objects through glass filters

### These filters have varying transmission as a function of wavelength (i.e. they are not step functions)

### Detectors sensitivity to photons (quantum efficiency) can also vary with wavelength 

### The standard UBVRI filters are shown below (taken from Chapter 5 Rieke)

<IMG SRC="UBVRI.png">

# Magnitudes measured through a filter

The flux measured through a filter is going to be 

\begin{equation}
F = \int f_\lambda \epsilon(\lambda) d\lambda
\end{equation}

where $\epsilon(\lambda)$ is the transmission curve of the filter. (This can also be written as a function of $\nu$.)

Given the defintion of a Vega based magnitude, this means 

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( \frac{ \int f_{\lambda,1} \epsilon(\lambda) d\lambda } { \int f_{\lambda,Vega} \epsilon(\lambda) d\lambda } \right)
\end{equation}

and we would expect

\begin{equation}
m_1 - 0 \simeq -2.5 {\rm log} ( f_{\lambda,1} / f_{\lambda,Vega} )
\end{equation}

when $f_{\lambda,1}$ and $f_{\lambda,Vega}$ are near the central wavelength of the filter. 

### Lets do a practical example of this now 

# Functions

### Functions allow us to package up particular tasks 

### This is particularly useful if tasks may need to be repeated many times 

### It can also limit the number of variables we need to consider simultaneously when writing and debugging code

### We will use some functions to calculate magnitudes 

###

In [1]:
import numpy as np

In [2]:
# Read a two column ascii file - only very slightly modified from what you can find on the internet

def Read_Two_Column_File(file_name):
    with open(file_name, 'r') as data:
        x = []   # Define an array with no elements
        y = []   # Define an array with no elements
        for line in data:
            if (line[0]!='#'):          # If the line is uncommented by a hash then
                p = line.split()        # Split the line components (i.e., the columns)
                x.append(float(p[0]))   # Add a new element to x
                y.append(float(p[1]))   # Add a new element to y

    return x, y

In [3]:
# Produce an interpolated flam (or transmission) - note this is a simple but very inefficient implementation

def Flam_interpolate(lam, star_lam, star_flam):
    flam = 0
    i=0
    if (lam>star_lam[0]) and (lam<star_lam[-1]):
        while star_lam[i]<lam:
            i = i + 1
        flam = (star_flam[i-1]*(star_lam[i]-lam) + star_flam[i]*(lam-star_lam[i-1])) / (star_lam[i]-star_lam[i-1])   
    return flam

In [4]:
def Flam_to_magnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran):

    # Initial values
    top=0.0
    bottom=0.0
    lam=fil_lam[0]
    dlam=1.0
    mag=-99.9

    # Step through lamba range of the filter
    while (lam<fil_lam[-1]):

        # Top line - star flux per unit lambda multiplied by filter transmisison and dlambda
        top = top + Flam_interpolate(lam,star_lam,star_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * dlam
        
        # Bottom line - Vega flux per unit lamdba multiplied by filter transmisson and dlambda
        bottom = bottom + Flam_interpolate(lam,vega_lam,vega_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * dlam
        
        # Increment lambda by delta lambda 
        lam = lam + dlam

    # If integrals have positive values return a magnitude that isn't -99
    if (top>0.0) and (bottom>0.0):
         mag = -2.5 * np.log10( top / bottom)

    # Return the magnitude value        
    return mag    


In [5]:
#  Vega ftp://ftp.eso.org/pub/stecf/standards/hststan/fhr7001.dat
vega_lam, vega_flam = Read_Two_Column_File('fhr7001.dat')    # Vega units Ang and ergs/cm/cm/s/A * 10**16
print(vega_lam[0], vega_flam[0])
print(vega_lam[1], vega_flam[1])
print(vega_lam[-1], vega_flam[-1])

1148.6 4231900.0
1149.8 10748000.0
10502.8838 5498600.0


In [6]:
# Star ftp://ftp.eso.org/pub/stecf/standards/hststan/
star_lam, star_flam = Read_Two_Column_File('ffeige34.dat')   # Star units Ang and erg/cm/cm/s/A * 10**16
print(star_lam[0], star_flam[0])
print(star_lam[1], star_flam[2])
print(star_lam[-1], star_flam[-1])

971.82 468540.0
981.08 384230.0
11993.0195 67.012


In [7]:
# Filter  V_bessell.dat 
fil_lam, fil_tran = Read_Two_Column_File('V_bessell.dat')  
print(fil_lam[0], fil_tran[0])
print(fil_lam[1], fil_tran[1])
print(fil_lam[-1], fil_tran[-1])
print(4750, Flam_interpolate(4750, fil_lam, fil_tran))

4700.0 0.0
4800.0 0.03
7000.0 0.0
4750 0.015


In [8]:
# Lets run function. Please note the wavelengths and flam need consistent units. Tranmission can be fractional or percentage.

mag=Flam_to_magnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran)
print('Magnitude:', mag)

Magnitude: 11.134336979407154


In [9]:
# Lets do the simple calculation where we just take flam values at the effective wavelength
# The effective wavelength of the V-band filter 5477 Angstroms so....
i = 0
while (star_lam[i]<5477):
    i = i + 1
j = 0
while (vega_lam[j]<5477):
    j = j + 1
print('Flam values:', star_flam[i], vega_flam[j])    
print('Approximate magnitude:', -2.5*np.log10(star_flam[i]/vega_flam[j]))   

Flam values: 1239.9 35404000.0
Approximate magnitude: 11.13916418018741


### Search online for a Fiege 34 V-band magnitude. How does it compare to what we have calculated from a spectrum?